# M03A: Structured Outputs & Validation

Free-form AI responses break production systems. Structured outputs fix that.

**Topics:**
- Getting JSON from AI
- Complex schemas and nested data
- Validation and verification patterns

---

## 🔧 Step 1: Setup

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
import openai

load_dotenv(dotenv_path=Path("..") / ".env")
client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-5-mini"

def ask_openai(prompt, model=MODEL):
    """Ask OpenAI a question using the global client."""
    
    try:
        response = client.responses.create(
            model=model,
            input=prompt
        )
        return response.output_text.strip()
        
    except openai.AuthenticationError:
        return "Error: Invalid API key. Check your .env file."
    except openai.RateLimitError:
        return "Error: Rate limit exceeded. Wait and try again."
    except openai.APIConnectionError:
        return "Error: Network issue. Check your internet."
    except openai.BadRequestError:
        return "Error: Bad request. Check model name."
    except Exception as e:
        return f"API Error: {str(e)}"


print(f"✅ Setup complete: Using {MODEL}!")

---

## 🔗 From Labels to Complex Data

Few-shot handles simple labels—positive, negative, neutral.

But production apps need to extract:
- Contact info from emails
- Product details from descriptions
- Multi-field feedback with routing

That requires structured outputs.

In [ ]:
# Few-shot prompting for sentiment (from M02B)
fewshot_sentiment = """Classify product review sentiment.

Examples:
Review: "Amazing quality, highly recommend!"
positive

Review: "Terrible product, stopped working after a week."
negative

Review: "It's okay, nothing special."
neutral

Now classify:
Review: "Great features but a bit pricey."
"""

print("📊 FEW-SHOT PROMPTING (Text Output)")
print("="*60)
print("Result: ", end="")
print(ask_openai(fewshot_sentiment))
print("\n" + "="*60)

✅ Few-shot works great for simple labels.

But production apps need:
- Multiple fields (sentiment + confidence + reason)
- Parsing and validation

---

## 🎯 Why Structured Outputs Matter

Text responses work for chat, but production apps need **parseable data**.

Let's see what happens when we try to extract contact information:

In [ ]:
# Extract contact info - text response
email_text = """Hi, I'm Sarah Johnson from Acme Corp. You can reach me at 
sarah.j@acmecorp.com or call me at (555) 123-4567. Looking forward to 
discussing the enterprise plan."""

prompt = f"""Extract the contact information from this email:

{email_text}

Extract: name, company, email, phone"""

print("📧 TEXT RESPONSE")
print("="*60)
result = ask_openai(prompt)
print(result)
print("\n" + "="*60)

### 📊 Analysis

✅ The model gave us clean output!

But for production:
- Format varies between calls
- Can't parse → can't automate

### 📌 Production Pattern

1. **Define schema** — What fields do you need?
2. **Prompt for JSON** — Ask the model to output that schema
3. **Parse** — Convert string to Python dict
4. **Validate** — Check required fields exist
5. **Verify** (optional) — Use a second AI call to check quality

---

## 💡 Solution: Request JSON Output

Ask for JSON format explicitly. Adding "return ONLY valid JSON" makes parsing easier.

### 🔧 JSON Parsing Helper

Models sometimes wrap JSON in markdown code fences. This helper strips them before parsing:

In [ ]:
def parse_json(text):
    """Parse JSON from API output, handling markdown fences."""
    clean = text.strip()
    
    # Strip markdown code fences if present
    if clean.startswith('```'):
        clean = clean.split('```')[1]
        if clean.startswith('json'):
            clean = clean[4:]
        clean = clean.strip()
    
    try:
        return json.loads(clean)
    except json.JSONDecodeError:
        return None

print("✅ parse_json() ready")

### Using the Helper

Now let's request JSON output and parse it with our helper:

In [ ]:
# Request JSON format
prompt = f"""Extract the contact information from this email and return as JSON.

Email:
{email_text}

Return JSON with these exact keys: name, company, email, phone

Return ONLY the JSON object, no markdown formatting, no explanation.
"""

print("📋 JSON RESPONSE")
print("="*60)
result = ask_openai(prompt)
print(result)
print("\n" + "="*60)

print("\n✅ Parsed Output:")
# Parse JSON string into Python dict
data = parse_json(result)

if data:
    print(f"   Name: {data['name']}")
    print(f"   Company: {data['company']}")
    print(f"   Email: {data['email']}")
    print(f"   Phone: {data['phone']}")

### 💡 Benefits of JSON Output

- **Access fields programmatically:** `data['email']`
- **Validate and integrate** with databases/APIs

---

## 📊 The Power of Combining Few-Shot + JSON

Combine few-shot prompting (for accuracy) with JSON output (for structure).

### Example: Sentiment Analysis with Multiple Fields

In [ ]:
# Few-shot prompting WITH JSON output
fewshot_json_prompt = """Analyze product reviews and return structured data.

Examples:
Review: "Amazing quality, highly recommend!"
{"sentiment": "positive", "confidence": 0.95, "key_points": ["quality", "recommendation"]}

Review: "Terrible product, stopped working after a week."
{"sentiment": "negative", "confidence": 0.90, "key_points": ["quality issues", "durability"]}

Review: "It's okay, nothing special."
{"sentiment": "neutral", "confidence": 0.85, "key_points": ["average"]}

Now analyze:
Review: "Great features but a bit pricey."
"""

print("🎯 FEW-SHOT + JSON (Best of Both Worlds)")
print("="*60)
result = ask_openai(fewshot_json_prompt)

# Parse into dict
data = parse_json(result)

if data:
    print(json.dumps(data, indent=2))
    print("\n✅ Parsed successfully:")
    print(f"   Sentiment: {data['sentiment']}")
    print(f"   Confidence: {data['confidence']*100:.0f}%")
    print(f"   Key points: {', '.join(data['key_points'])}")

### ✨ Benefits

- **High accuracy** from few-shot examples
- **Structured output** with multiple fields

### 📈 Comparison Table

<div style="text-align: left; display: inline-block;">

| Approach | Results | Format Consistency | Production Ready? |
|----------|---------|-------------------|------------------|
| Zero-shot + text | Variable | Low | ❌ |
| Few-shot + text | Good | Medium | ⚠️ |
| Zero-shot + JSON | Good | High | ⚠️ |
| **Few-shot + JSON** | **Excellent** | **High** | **✅** |

</div>

---

## 🔍 JSON Schemas: Define Your Structure

For complex data, define a clear schema in your prompt. This helps the model understand exactly what structure you need.

In [ ]:
# Define a schema for contact extraction
CONTACT_SCHEMA = {
    "name": "string",
    "company": "string",
    "email": "string",
    "phone": "string"
}

# More complex email with multiple contacts
complex_email = """Hi team,
I'm Sarah Johnson from Acme Corp (sarah.j@acmecorp.com, 555-123-4567).
Please also CC my colleague Mike Chen at mike.chen@acmecorp.com.
Looking forward to the demo!"""

prompt = f"""Extract ALL contact information from this email.

Return a JSON array of contact objects. Each contact object should follow this schema:
{json.dumps(CONTACT_SCHEMA, indent=2)}

Email:
{complex_email}

Return only valid JSON array. No markdown formatting, no explanation.
"""

print("📇 SCHEMA-BASED EXTRACTION")
print("="*60)
result = ask_openai(prompt)
print(result)
print("\n" + "="*60)

# Parse using helper (handles array output too)
contacts = parse_json(result)

if contacts:
    print("\n✅ Successfully parsed:")
    for i, contact in enumerate(contacts, 1):
        print(f"\n   Contact {i}:")
        print(f"   - Name: {contact.get('name', 'N/A')}")
        print(f"   - Company: {contact.get('company', 'N/A')}")
        print(f"   - Email: {contact.get('email', 'N/A')}")
        print(f"   - Phone: {contact.get('phone', 'N/A')}")

---

## 🏗️ Nested Data Structures

Real-world applications often need nested JSON. Let's extract product details with complex structure:

In [ ]:
# Product listing with nested data
product_text = """New Product Launch: SmartWidget Pro
Price: $299 (was $399, save $100)
Available colors: Black, White, Silver
Specs: 5-inch display, 128GB storage, 12-hour battery
Ships in: 2-3 business days
Rating: 4.5 stars (127 reviews)
"""

PRODUCT_SCHEMA = {
    "name": "string",
    "price": {
        "current": "number",
        "original": "number",
        "discount": "number"
    },
    "colors": ["array of strings"],
    "specs": {
        "display": "string",
        "storage": "string",
        "battery": "string"
    },
    "shipping": "string",
    "rating": {
        "stars": "number",
        "reviews": "number"
    }
}

prompt = f"""Extract product information and return as JSON.

Schema:
{json.dumps(PRODUCT_SCHEMA, indent=2)}

Product listing:
{product_text}

Return only valid JSON, no markdown formatting, no explanation.
"""

print("🛍️ NESTED DATA EXTRACTION")
print("="*60)
result = ask_openai(prompt)
print(result)
print("\n" + "="*60)

product_data = parse_json(result)

if product_data:
    print("\n✅ Successfully parsed nested JSON!")
    
    print("\n📊 Accessing nested data:")
    try:
        print(f"   Current price: ${product_data['price']['current']}")
        print(f"   Discount: ${product_data['price']['discount']} off")
        print(f"   Colors: {', '.join(product_data['colors'])}")
        print(f"   Storage: {product_data['specs']['storage']}")
        print(f"   Rating: {product_data['rating']['stars']} stars ({product_data['rating']['reviews']} reviews)")
    except KeyError as e:
        print(f"   ❌ Missing field in data: {e}")

---

## ✔️ Validation: Check Your Data

Getting JSON isn't enough - you need to validate it has the fields you expect.

In [ ]:
def validate_contact(data):
    """
    Validate contact data has required fields and correct types.
    
    Returns: (is_valid, message)
    """
    required_fields = ['name', 'company', 'email', 'phone']
    
    # Check all required fields exist
    for field in required_fields:
        if field not in data:
            return False, f"Missing required field: {field}"
        
        # Allow string or number/int (some AI models extract phone as int)
        if not isinstance(data[field], (str, int)):
            return False, f"Field {field} must be a string or number"

        # If it is a string, check it isn't empty
        if isinstance(data[field], str) and not data[field].strip():
            return False, f"Field {field} cannot be empty"
    
    # Basic email validation
    if '@' not in data['email']:
        return False, "Email must contain @"
    
    return True, "Valid"


# Test validation
contact_cases = [
    {"name": "John Doe", "company": "TechCorp", "email": "john@techcorp.com", "phone": "555-0100"},
    {"name": "Jane Smith", "company": "DevInc", "email": "invalid-email", "phone": "555-0200"},
    {"name": "Bob Wilson", "company": "StartupCo", "phone": "555-0300"},  # Missing email
]

print("🔍 VALIDATION EXAMPLES")
print("="*60)

for i, contact in enumerate(contact_cases, 1):
    is_valid, message = validate_contact(contact)
    status = "✅" if is_valid else "❌"
    print(f"\nTest {i}: {status}")
    print(f"Data: {contact}")
    print(f"Result: {message}")

---

## 🔬 The Verifier Pattern: AI Quality Checking

A second AI call checks the quality of the first one.

### How It Works

1. **Extract** — AI extracts structured data from input
2. **Verify** — Second AI call validates the extraction
3. **Score** — Verifier gives quality score (0-100)
4. **Decide** — Use data if score is high, retry if low

### 🔬 Example: Contact Extraction with Verification

In [ ]:
def verify_extraction(original_text, extracted_data):
    """Use AI to verify extraction quality. Returns dict with quality_score, issues, is_acceptable."""
    
    verify_prompt = f"""You are a quality checker. Verify if contact information was extracted correctly.

Original text:
{original_text}

Extracted data:
{json.dumps(extracted_data, indent=2)}

Rate the extraction quality (0-100) and explain any issues.

Return JSON with: {{"quality_score": 0-100, "issues": ["list of issues"], "is_acceptable": true/false}}

A score of 80+ means acceptable. Below 80 means retry extraction.

Return only valid JSON, no markdown formatting.
"""
    
    response = ask_openai(verify_prompt)
    data = parse_json(response)
    
    if data:
        return data
    else:
        return {"quality_score": 0, "issues": ["Verifier failed to parse"], "is_acceptable": False}

print("✅ verify_extraction() ready")

#### Test with Good Extraction

In [ ]:
good_email = "Hi, I'm Sarah Johnson from Acme Corp. Email: sarah@acme.com, Phone: 555-1234"
good_extraction = {
    "name": "Sarah Johnson",
    "company": "Acme Corp",
    "email": "sarah@acme.com",
    "phone": "555-1234"
}

print("🔍 VERIFIER PATTERN - Good Extraction")
print("="*60)
verification = verify_extraction(good_email, good_extraction)
print(f"Quality score: {verification['quality_score']}/100")
print(f"Acceptable: {'✅ Yes' if verification['is_acceptable'] else '❌ No'}")
if verification['issues']:
    print(f"Issues: {', '.join(verification['issues'])}")

#### Test with Bad Extraction

In [ ]:
bad_extraction = {
    "name": "Sarah",  # Missing last name
    "company": "Acme Corp",
    "email": "sarah@acme.com",
    "phone": ""  # Missing phone
}

print("🔍 VERIFIER PATTERN - Poor Extraction")
print("="*60)
verification = verify_extraction(good_email, bad_extraction)
print(f"Quality score: {verification['quality_score']}/100")
print(f"Acceptable: {'✅ Yes' if verification['is_acceptable'] else '❌ No'}")
if verification['issues']:
    print(f"Issues:")
    for issue in verification['issues']:
        print(f"   - {issue}")

---

### 🏭 Complete Production Pattern

Here's the full extract → validate → verify → use pattern:

In [ ]:
def extract_with_verification(email_text):
    """
    Production-grade extraction with validation and verification.
    
    Returns: (success, data, message)
    """
    # Step 1: Extract
    extract_prompt = f"""Extract contact information and return as JSON.

Email:
{email_text}

Return JSON with these exact keys: name, company, email, phone

Return only valid JSON, no markdown formatting.
"""
    
    response = ask_openai(extract_prompt)
    
    # Check for API errors first
    if response.startswith("Error:"):
        return False, None, response

    # Step 2: Parse
    data = parse_json(response)
    if not data:
        return False, None, "JSON parse error"
    
    # Step 3: Validate
    is_valid, validation_msg = validate_contact(data)
    if not is_valid:
        return False, data, f"Validation error: {validation_msg}"
    
    # Step 4: Verify (optional for critical data)
    verification = verify_extraction(email_text, data)
    if not verification['is_acceptable']:
        issues = ', '.join(verification['issues'])
        return False, data, f"Quality check failed: {issues}"
    
    return True, data, "Success"

print("✅ extract_with_verification() ready")

#### Test the full pipeline

In [ ]:
test_email = """Hey there! I'm Mike Chen, CTO at DevStartup Inc. 
Best way to reach me is mike.chen@devstartup.io or my cell: (555) 987-6543.
Let's schedule that demo!"""

print("🏭 COMPLETE PRODUCTION PIPELINE")
print("="*60)
print(f"Input: {test_email[:60]}...\n")

success, data, message = extract_with_verification(test_email)

if success:
    print("✅ Extraction successful!")
    print("\nExtracted data:")
    print(json.dumps(data, indent=2))
else:
    print(f"❌ Extraction failed: {message}")
    if data:
        print("\nPartial data:")
        print(json.dumps(data, indent=2))

### 💡 When to Use Verifier Pattern

**Use for:** Critical data, high-stakes decisions, costly errors

**Skip for:** Low-stakes apps, speed-sensitive, budget-constrained

**Trade-off:** Double the API calls, but higher reliability

---

## 🔒 The Production Way: Schema Enforcement

Everything above works — but it depends on the model following your instructions.  

The Responses API has a `text` parameter that enforces your schema server-side.  

Guaranteed valid JSON, no parsing needed.

In [ ]:
# Schema enforcement via text parameter
response = client.responses.create(
    model=MODEL,
    input=f'Extract contact information from this email:\n\n{email_text}',
    text={
        "format": {
            "type": "json_schema",
            "name": "contact",
            "strict": True,
            "schema": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "name": {"type": "string"},
                    "company": {"type": "string"},
                    "email": {"type": "string"},
                    "phone": {"type": "string"}
                },
                "required": ["name", "company", "email", "phone"]
            }
        }
    }
)

# No parse_json() needed — guaranteed valid
data = json.loads(response.output_text.strip())

print("🔒 SCHEMA ENFORCEMENT")
print("="*60)
print(f"Name:    {data['name']}")
print(f"Company: {data['company']}")
print(f"Email:   {data['email']}")
print(f"Phone:   {data['phone']}")
print("="*60)

### 💡 When to Use Each Approach

- **Prompt-based JSON** — Quick prototyping, simple schemas, any API
- **`text` parameter** — Production systems, strict validation, guaranteed output

---

## 📋 Complete Example: Customer Feedback Analyzer

Schema enforcement + few-shot + business logic working together.

In [ ]:
# Complete example: Customer feedback analyzer
sample_feedback = """Your app keeps crashing when I try to export my data. This is 
really frustrating because I need this for a client presentation tomorrow. 
Otherwise the app is great, but this bug is a showstopper."""

response = client.responses.create(
    model=MODEL,
    input=f"""Analyze customer feedback and extract structured information.

Example:
Feedback: "Love the new dark mode feature! Makes it easier to work at night."
{{"sentiment": "positive", "category": "praise", "urgency": "low", "key_points": ["dark mode", "usability"], "requires_response": false}}

Now analyze this feedback:
{sample_feedback}""",
    text={
        "format": {
            "type": "json_schema",
            "name": "feedback_analysis",
            "strict": True,
            "schema": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "sentiment": {"type": "string", "enum": ["positive", "negative", "neutral"]},
                    "category": {"type": "string", "enum": ["bug", "feature", "question", "complaint", "praise"]},
                    "urgency": {"type": "string", "enum": ["high", "medium", "low"]},
                    "key_points": {"type": "array", "items": {"type": "string"}},
                    "requires_response": {"type": "boolean"}
                },
                "required": ["sentiment", "category", "urgency", "key_points", "requires_response"]
            }
        }
    }
)

# No parse_json() needed — guaranteed valid
feedback_data = json.loads(response.output_text.strip())

print("📝 CUSTOMER FEEDBACK ANALYZER")
print("="*60)
print(f"Feedback: {sample_feedback}\n")
print("="*60 + "\n")

print("✅ Successfully parsed:")
print(f"   Sentiment: {feedback_data['sentiment']}")
print(f"   Category: {feedback_data['category']}")
print(f"   Urgency: {feedback_data['urgency']}")
print(f"   Key points: {', '.join(feedback_data['key_points'])}")
print(f"   Requires response: {'Yes' if feedback_data['requires_response'] else 'No'}")

# Business logic based on structured data
print("\n🎯 Automated routing:")
if feedback_data['urgency'] == 'high' and feedback_data['category'] == 'bug':
    print("   → Route to: Engineering team (high priority)")
    print("   → SLA: Response within 4 hours")
elif feedback_data['category'] == 'praise':
    print("   → Route to: Success team (track satisfaction)")
else:
    print("   → Route to: Support team (standard queue)")

### 💡 Why This Works

- **Few-shot example** shows the model what good output looks like
- **Schema enforcement** guarantees every field exists with a valid value
- **Safe to use in code** — you can route on `urgency` and `category` without worrying about missing keys or unexpected values

---

### 💪 Your Turn: Build Your Own Analyzer

Now try building your own feedback analyzer with different categories or fields:

In [ ]:
# Exercise: Build Your Own Analyzer
# Objective: Create a custom JSON extractor for "Job Application Emails"
# using the text parameter for schema enforcement.

sample_email = """
Hi, I'm Alex Chen. I'd like to apply for the Senior Python Developer role.
I have 8 years of experience in backend development, specifically with Django and AWS.
I also have some frontend React skills. You can reach me at alex@example.com.
"""

print("=== Exercise: Job Application Parser ===\n")

# TODO: Fill in the four properties below, then run the cell.
#   - name: string
#   - role: string
#   - years_experience: integer
#   - key_skills: array of strings

response = client.responses.create(
    model=MODEL,
    input=f'Extract job application details from this email:\n\n{sample_email}',
    text={
        "format": {
            "type": "json_schema",
            "name": "job_application",
            "strict": True,
            "schema": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    # TODO: Add your four properties here
                },
                "required": ["name", "role", "years_experience", "key_skills"]
            }
        }
    }
)

data = json.loads(response.output_text.strip())
print(f"Name: {data['name']}")
print(f"Role: {data['role']}")
print(f"Experience: {data['years_experience']} years")
print(f"Skills: {', '.join(data['key_skills'])}")

---

## 🎯 Key Takeaways

**Prompt-Based JSON:**
- Ask for JSON in the prompt and parse with `json.loads()`
- Fast for prototyping, but the model can still return invalid JSON
- Use `parse_json()` helper to handle failures gracefully

**Few-Shot + JSON:**
- Few-shot guides the model's judgment (what to output)
- JSON enforces the format (how to output it)
- Combine both for accuracy and consistency

**The Verifier Pattern:**
- Use a second AI call to score output quality (0-100)
- Trade-off: better quality vs. extra cost and latency

**Schema Enforcement (`text` parameter):**
- Guarantees valid JSON every time — no parsing failures
- Use `enum` to constrain values to a fixed set
- Use for production where reliability matters

---

### 📍 Next Step

**M03B: Error Handling & Retry Logic** — API failures, exponential backoff, and fault tolerance.

---

## 🔧 Troubleshooting

**AI not returning valid JSON?**
- Be explicit: "Return ONLY valid JSON, no markdown"
- Use the `parse_json()` helper to strip markdown fences
- Print raw response to debug formatting

**Validation keeps failing?**
- Check if schema is too strict (use `.get()`)
- Log the raw output to see what went wrong

**Extraction quality low?**
- Add few-shot examples showing correct JSON
- Make prompt instructions/schema more explicit

**Verifier pattern too slow?**
- Make it optional (only for critical data)
- Use for spot-checking rather than every call

**Schema enforcement not working?**
- Verify `strict` is set to `True`
- Check that `additionalProperties` is `False`
- Ensure all fields are listed in `required`

**Still having issues?**
- Copy any error message and paste it into ChatGPT, Claude, Gemini, or Grok — they're great at debugging
- Re-watch the lecture for this module
- Post to the Q&A with your error message and output

---